# California Housing - Stacking từng bước (chi tiết)



In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVR

In [2]:
SEED = 42
DATA_PATH = "./data/housing.csv"

TARGET_COL = "median_house_value"
LOG_NUM_COLS = ["total_rooms", "total_bedrooms", "households", "population"]
NUM_COLS = [
    "longitude", "latitude", "housing_median_age", "median_income",
    "rooms_per_household", "population_per_household", "bedrooms_per_room",
]
CAT_COLS = ["ocean_proximity", "geo_cluster"]

## 1) Load data

In [3]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (20640, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## 2) Train/Test split + target log scale

In [4]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)

y_train_log = np.log1p(train_df[TARGET_COL].to_numpy(dtype=float))
y_test = test_df[TARGET_COL].to_numpy(dtype=float)

print("Train/Test:", train_df.shape, test_df.shape)

Train/Test: (16512, 10) (4128, 10)


## 3) Feature Engineering 

In [5]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self, use_geo_cluster=True, n_geo_clusters=10, random_state=SEED):
        self.use_geo_cluster = use_geo_cluster
        self.n_geo_clusters = n_geo_clusters
        self.random_state = random_state
        self.kmeans_ = None

    def fit(self, X, y=None):
        X = X.copy()
        if self.use_geo_cluster:
            self.kmeans_ = KMeans(
                n_clusters=self.n_geo_clusters,
                random_state=self.random_state,
                n_init=10,
            )
            self.kmeans_.fit(X[["longitude", "latitude"]])
        return self

    def transform(self, X):
        X = X.copy()

        X["ocean_proximity"] = X["ocean_proximity"].replace("ISLAND", "NEAR OCEAN")

        households = X["households"].replace(0, 1)
        total_rooms = X["total_rooms"].replace(0, 1)
        X["rooms_per_household"] = X["total_rooms"] / households
        X["population_per_household"] = X["population"] / households
        X["bedrooms_per_room"] = X["total_bedrooms"] / total_rooms

        if self.use_geo_cluster and self.kmeans_ is not None:
            labels = self.kmeans_.predict(X[["longitude", "latitude"]])
            X["geo_cluster"] = labels.astype(str)
        else:
            X["geo_cluster"] = "0"

        return X

In [6]:
fe = FeatureEngineering(use_geo_cluster=True, n_geo_clusters=10, random_state=SEED)
fe.fit(train_df)

train_fe = fe.transform(train_df)
test_fe = fe.transform(test_df)

train_X_df = train_fe.drop(columns=[TARGET_COL])
test_X_df = test_fe.drop(columns=[TARGET_COL])

## 4) Preprocess 

In [7]:
def build_preprocessor():
    log_num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, validate=False)),
        ("scaler", StandardScaler()),
    ])

    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    cat_pipeline = Pipeline([
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer([
        ("log_num", log_num_pipeline, LOG_NUM_COLS),
        ("num", num_pipeline, NUM_COLS),
        ("cat", cat_pipeline, CAT_COLS),
    ])
    return preprocessor

In [8]:
preprocessor = build_preprocessor()
X_train = preprocessor.fit_transform(train_X_df)
X_test = preprocessor.transform(test_X_df)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (16512, 25)
X_test shape: (4128, 25)


## hàm trong Stacking



In [9]:
def build_base_models(seed=SEED):
    return {
        "ridge": Ridge(alpha=1.0, random_state=seed),
        "rf": RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_split=5,
            random_state=seed,
            n_jobs=-1,
        ),
        "svr": SVR(C=30.0, epsilon=0.1, gamma="scale"),
    }


def build_meta_model(seed=SEED):
    return Ridge(alpha=0.5, random_state=seed)

In [10]:
def generate_oof_predictions(X, y, base_models, n_splits=5, seed=SEED):

    X = np.asarray(X)
    y = np.asarray(y)
    model_names = list(base_models.keys())

    oof = np.zeros((X.shape[0], len(model_names)), dtype=float)
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for fold_idx, (tr_idx, va_idx) in enumerate(cv.split(X), start=1):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_va = X[va_idx]

        for j, name in enumerate(model_names):
            model = clone(base_models[name])
            model.fit(X_tr, y_tr)
            oof[va_idx, j] = model.predict(X_va)

    return oof, model_names


def fit_base_models_on_full_data(X, y, base_models, model_names):
    full_models = {}
    for name in model_names:
        model = clone(base_models[name])
        model.fit(X, y)
        full_models[name] = model
    return full_models


def fit_meta_model(oof_matrix, y, meta_model):
    meta = clone(meta_model)
    meta.fit(oof_matrix, y)
    return meta


def create_meta_features_from_base_models(X, full_base_models, model_names):
    meta_features = []
    for name in model_names:
        preds = full_base_models[name].predict(X)
        meta_features.append(preds.reshape(-1, 1))
    return np.hstack(meta_features)


def stacking_predict(X, full_base_models, meta_model, model_names):
    meta_X = create_meta_features_from_base_models(X, full_base_models, model_names)
    return meta_model.predict(meta_X)

## 6) Train Stacking thủ công

In [11]:
base_models = build_base_models(seed=SEED)
meta_model = build_meta_model(seed=SEED)

oof_matrix, model_names = generate_oof_predictions(
    X=X_train,
    y=y_train_log,
    base_models=base_models,
    n_splits=5,
    seed=SEED,
)

full_base_models = fit_base_models_on_full_data(
    X=X_train,
    y=y_train_log,
    base_models=base_models,
    model_names=model_names,
)

fitted_meta_model = fit_meta_model(
    oof_matrix=oof_matrix,
    y=y_train_log,
    meta_model=meta_model,
)

print("OOF shape:", oof_matrix.shape)
print("Base models:", model_names)

OOF shape: (16512, 3)
Base models: ['ridge', 'rf', 'svr']


## 7) Predict + Evaluate 

In [12]:
pred_log = stacking_predict(
    X=X_test,
    full_base_models=full_base_models,
    meta_model=fitted_meta_model,
    model_names=model_names,
)

pred = np.expm1(pred_log)

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("=== Manual Function Stacking (Original Scale) ===")
print(f"RMSE: {rmse:,.2f}")
print(f"MAE : {mae:,.2f}")
print(f"R2  : {r2:.4f}")

=== Manual Function Stacking (Original Scale) ===
RMSE: 48,564.42
MAE : 30,253.98
R2  : 0.8200
